# ✅ Final Bug-Free Customer Churn Pipeline

## 1. Upload Files

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from google.colab import files
    uploaded = files.upload()
except:
    print('Running locally')

print(os.listdir())

## 2. Auto Detect Files

In [ ]:
def find_file(keys):
    for f in os.listdir():
        if all(k in f.lower() for k in keys):
            return f
    return None

files_map = {
 'customers':['customers'],
 'orders':['orders'],
 'support':['support'],
 'web':['web'],
 'churn':['churn'],
 'campaign':['intervention']
}

detected = {k:find_file(v) for k,v in files_map.items()}
print(detected)

## 3. Load Data

In [ ]:
customers = pd.read_csv(detected['customers'])
orders = pd.read_csv(detected['orders'])
support = pd.read_csv(detected['support'])
web = pd.read_csv(detected['web'])
churn = pd.read_csv(detected['churn'])
campaign = pd.read_csv(detected['campaign'])

## 4. Feature Engineering

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
churn['snapshot_date'] = pd.to_datetime(churn['snapshot_date'])

order_agg = orders.groupby('customer_id').agg({
 'order_id':'count',
 'gross_amount':['sum']
}).reset_index()
order_agg.columns = ['customer_id','total_orders','total_spend']

last_order = orders.groupby('customer_id')['order_date'].max().reset_index()
last_order.columns = ['customer_id','last_order_date']
last_order = last_order.merge(churn[['customer_id','snapshot_date']])
last_order['recency_days'] = (last_order['snapshot_date'] - last_order['last_order_date']).dt.days
order_agg = order_agg.merge(last_order[['customer_id','recency_days']])

support_agg = support.groupby('customer_id').agg({
 'ticket_id':'count',
 'sentiment_score':'mean'
}).reset_index()
support_agg.columns = ['customer_id','ticket_count','avg_sentiment']

# ✅ FIX: Campaign feature
campaign['received_campaign'] = campaign['last_campaign_received'].apply(lambda x: 0 if str(x).lower()=='none' else 1)
campaign['received_campaign'] = campaign['received_campaign'].fillna(0)

df_final = churn.merge(customers,on='customer_id',how='left')
df_final = df_final.merge(order_agg,on='customer_id',how='left')
df_final = df_final.merge(support_agg,on='customer_id',how='left')
df_final = df_final.merge(web,on='customer_id',how='left')
df_final = df_final.merge(campaign,on='customer_id',how='left')

print(df_final.columns)

## 5. Visualization

In [ ]:
sns.set(style='whitegrid')

required_cols = ['received_campaign','churn_next_60d']
missing = [c for c in required_cols if c not in df_final.columns]
if missing:
    print('Missing:', missing)

plt.figure(figsize=(6,4))
sns.countplot(x='churn_next_60d', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='total_orders', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='total_spend', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='recency_days', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='ticket_count', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='avg_sentiment', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='sessions_30d', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x='churn_next_60d', y='last_visit_days_ago', data=df_final)
plt.show()

plt.figure(figsize=(8,5))
sns.countplot(x='loyalty_tier', hue='churn_next_60d', data=df_final)
plt.show()

plt.figure(figsize=(6,4))
sns.countplot(x='received_campaign', hue='churn_next_60d', data=df_final)
plt.show()